# 01 - Data audit

## Objective
Inspect the raw Telco Churn dataset, understand its structure,
identify data-quality problems, and define the target variable.

This audit will evaluate:

- Dataset structure and dimensions
- Column names and data types
- Missing and duplicated values
- Unique and potentially invalid values
- Target-class distribution
- Potential data leakage risks
- Data-quality decisions required before modeling

In [182]:
from pathlib import Path
import pandas as pd

In [156]:
Path.cwd()
PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_PATH = DATA_DIR / "raw" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"

In [157]:
if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found at {RAW_DATA_PATH.resolve()}")

In [158]:
dataset=pd.read_csv(RAW_DATA_PATH)

## Initial observations

- Each row represents one telecommunications customer.
- The dataset contains customer demographics, subscribed services, account information, billing information, and the churn target.
- Most service-related variables appear to be categorical.
- `customerID` appears to be a unique customer identifier rather than a predictive customer characteristic.
- `Churn` is the target variable and indicates whether a customer left the company.

In [159]:
dataset.head(8)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
5,9305-CDSKC,Female,0,No,No,8,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.5,Yes
6,1452-KIOVK,Male,0,No,Yes,22,Yes,Yes,Fiber optic,No,...,No,No,Yes,No,Month-to-month,Yes,Credit card (automatic),89.10,1949.4,No
7,6713-OKOMC,Female,0,No,No,10,No,No phone service,DSL,Yes,...,No,No,No,No,Month-to-month,No,Mailed check,29.75,301.9,No


In [160]:
dataset.sample(8,random_state=5)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
4213,3683-QKIUE,Female,0,No,No,6,Yes,No,DSL,No,...,No,Yes,No,No,Month-to-month,Yes,Bank transfer (automatic),50.80,288.05,Yes
5035,3955-JBZZM,Male,0,No,No,20,Yes,No,Fiber optic,No,...,No,No,Yes,No,Month-to-month,No,Electronic check,78.80,1641.3,No
3713,6961-VCPMC,Male,1,Yes,No,46,Yes,No,Fiber optic,No,...,No,No,No,Yes,Month-to-month,Yes,Electronic check,80.40,3605.2,Yes
1720,6407-UTSLV,Female,1,No,No,2,Yes,No,Fiber optic,No,...,No,No,Yes,No,Month-to-month,Yes,Bank transfer (automatic),83.80,163.7,No
234,1984-GPTEH,Female,0,No,No,29,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Electronic check,25.15,702,No
4558,8063-GBATB,Female,1,No,No,27,Yes,Yes,Fiber optic,No,...,No,No,No,Yes,Month-to-month,Yes,Electronic check,85.25,2287.25,Yes
40,8865-TNMNX,Male,0,Yes,Yes,10,Yes,No,DSL,No,...,No,No,No,No,One year,No,Mailed check,49.55,475.7,No
3455,7274-RTAPZ,Male,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,90.55,90.55,Yes


## Dataset dimensions

The raw dataset contains 7,043 observations and 21 columns.

Each observation represents one customer. The columns include the target variable, customer characteristics, service information, billing variables, and a customer identifier.

In [161]:
n_rows,n_columns = dataset.shape
print(f"number of Rows: {n_rows}\nnumber of columns: {n_columns}")

number of Rows: 7043
number of columns: 21


## Data type observations

- The dataset contains 18 string columns, 2 integer columns, and 1 floating-point column.
- Most variables are categorical and are currently represented using the pandas string data type.
- `tenure` and `SeniorCitizen` are stored as integers.
- `MonthlyCharges` is stored as a floating-point variable.
- `TotalCharges` is stored as a string even though it appears to represent a numeric amount.

In [162]:
dataset.dtypes

customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object

In [163]:
dataset.dtypes.value_counts()

str        18
int64       2
float64     1
Name: count, dtype: int64

## Column groups

Based on the current pandas data types:

- Numeric columns were identified using `select_dtypes(include="number")`.
- Categorical columns were identified using `select_dtypes(include="string")`.
- The current grouping is based on storage type, not necessarily the true semantic meaning of each variable.

In [164]:
numeric_columns = dataset.select_dtypes(include="number").columns.tolist()

categorical_columns = dataset.select_dtypes(include=["string"]).columns.tolist()

print("Numeric columns:")
print(numeric_columns)

print("\nCategorical columns:")
print(categorical_columns)

Numeric columns:
['SeniorCitizen', 'tenure', 'MonthlyCharges']

Categorical columns:
['customerID', 'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'TotalCharges', 'Churn']


In [165]:
column_audit = pd.DataFrame(
    {
        "column": dataset.columns,
        "dtype": dataset.dtypes.astype(str).values,
        "unique_count": dataset.nunique(dropna=False).values,
    }
)

column_audit["possible_role"] = column_audit.apply(
    lambda row: (
        "identifier"
        if row["column"] == "customerID"
        else "binary categorical"
        if row["unique_count"] == 2
        else "numeric"
        if row["dtype"] in ["int64", "float64"]
        else "multiclass categorical"
    ),
    axis=1,
)

column_audit

,column,dtype,unique_count,possible_role
0,customerID,str,7043,identifier
1,gender,str,2,binary categorical
2,SeniorCitizen,int64,2,binary categorical
3,Partner,str,2,binary categorical
4,Dependents,str,2,binary categorical
5,tenure,int64,73,numeric
6,PhoneService,str,2,binary categorical
7,MultipleLines,str,3,multiclass categorical
8,InternetService,str,3,multiclass categorical
9,OnlineSecurity,str,3,multiclass categorical


In [166]:
column_audit.loc[
    column_audit["column"] == "TotalCharges",
    "possible_role"
] = "numeric stored as text"

column_audit.loc[
    column_audit["column"] == "Churn",
    "possible_role"
] = "target"

column_audit.loc[
    column_audit["column"] == "SeniorCitizen",
    "possible_role"
] = "binary categorical stored as integer"

column_audit

,column,dtype,unique_count,possible_role
0,customerID,str,7043,identifier
1,gender,str,2,binary categorical
2,SeniorCitizen,int64,2,binary categorical stored as integer
3,Partner,str,2,binary categorical
4,Dependents,str,2,binary categorical
5,tenure,int64,73,numeric
6,PhoneService,str,2,binary categorical
7,MultipleLines,str,3,multiclass categorical
8,InternetService,str,3,multiclass categorical
9,OnlineSecurity,str,3,multiclass categorical


## Missing Values

This section checks for standard missing values recognized by pandas, such as `NaN` or `None`.

For each column, the analysis reports:

- the number of missing values;
- the percentage of missing values relative to the total number of rows.

Columns without missing values are excluded from the summary to keep the output focused.

In [167]:
missing_values = dataset.isna().sum()

missing_summary = pd.DataFrame(
    {
        "column": missing_values.index,
        "missing_count": missing_values.values,
        "missing_percentage": (
            missing_values.values / len(dataset) * 100
        ),
    }
)

missing_summary = missing_summary[
    missing_summary["missing_count"] > 0
]

missing_summary

,column,missing_count,missing_percentage


### Finding

No standard missing values were detected by `pandas.isna()`.

However, this result does not guarantee that the dataset is complete. Missing information may also be represented as blank strings, spaces, placeholder text, or invalid values. Therefore, categorical columns must also be checked for blank strings.

### Blank Strings in Categorical Columns

Categorical columns are inspected for empty strings or strings containing only whitespace.

This validation is necessary because blank strings are not automatically recognized as missing values by `pandas.isna()`.

In [168]:
blank_string_summary = []

for column in categorical_columns:
    blank_count = (
        dataset[column]
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )

    blank_string_summary.append(
        {
            "column": column,
            "blank_string_count": blank_count,
        }
    )

blank_string_summary = pd.DataFrame(blank_string_summary)

blank_string_summary[
    blank_string_summary["blank_string_count"] > 0
]

,column,blank_string_count
16,TotalCharges,11


### Finding

The `TotalCharges` column contains 11 blank-string values.

Although `TotalCharges` is currently stored as a categorical column, it represents a numeric amount. These blank values must therefore be investigated before converting the column to a numeric data type.

## Duplicate Records

This section checks whether the dataset contains fully duplicated rows.

Duplicated records may distort distributions, bias model training, and cause the same observation to receive excessive influence.

In [169]:
duplicate_count = dataset.duplicated().sum()

print(f"Duplicated rows: {duplicate_count}")

Duplicated rows: 0


### Finding

No fully duplicated rows were found in the dataset.

### Duplicate Customer Identifiers

The `customerID` column is expected to uniquely identify each customer.

This check verifies that no customer appears more than once under the same identifier, even if the remaining column values differ.

In [170]:
duplicated_customer_ids = dataset["customerID"].duplicated().sum()

print(f"Duplicated customer IDs: {duplicated_customer_ids}")

Duplicated customer IDs: 0


### Finding

No duplicated customer identifiers were found. Each row represents a unique customer.

## Invalid Numeric Values

The `TotalCharges` column represents a numeric amount but is currently stored as text.

To identify invalid entries, the column is temporarily converted with `pd.to_numeric(..., errors="coerce")`. Any value that cannot be interpreted as a number is converted to `NaN` and counted as invalid.

In [171]:
total_charges_numeric = pd.to_numeric(
    dataset["TotalCharges"],
    errors="coerce"
)

invalid_total_charges = total_charges_numeric.isna().sum()

print(
    f"Values in TotalCharges that cannot be converted: "
    f"{invalid_total_charges}"
)

Values in TotalCharges that cannot be converted: 11


### Finding

A total of 11 values in `TotalCharges` cannot be converted to numeric format.

These are the same records previously identified as blank strings.

### Investigation of Invalid `TotalCharges` Records

The affected records are inspected together with `tenure` and `MonthlyCharges` to determine whether the missing values follow a meaningful pattern.

In [172]:
dataset.loc[
    total_charges_numeric.isna(),
    ["customerID", "tenure", "MonthlyCharges", "TotalCharges"]
]

,customerID,tenure,MonthlyCharges,TotalCharges
488,4472-LVYGI,0,52.55,
753,3115-CZMZD,0,20.25,
936,5709-LVOEQ,0,80.85,
1082,4367-NUYAO,0,25.75,
1340,1371-DWPAZ,0,56.05,
3331,7644-OMVMY,0,19.85,
3826,3213-VVOLG,0,25.35,
4380,2520-SGTTA,0,20.00,
5218,2923-ARZLG,0,19.70,
6670,4075-WKNIU,0,73.35,


### Interpretation

All customers with invalid `TotalCharges` values have `tenure = 0`.

This suggests that they are new customers who have not yet completed a billing period. Therefore, the blank value is likely caused by the business process rather than by random data corruption.

A reasonable preprocessing decision is to convert `TotalCharges` to numeric and replace these missing values with `0`, because customers with zero tenure have not accumulated historical charges.

## Target Distribution

This section examines the distribution of the target variable, `Churn`.

The objective is to determine how many customers belong to each target class and whether the classification problem is balanced or imbalanced.

Class imbalance can affect model training and make accuracy misleading, especially if the minority class represents the business event of interest.

In [175]:
target_counts = dataset["Churn"].value_counts()

target_percentages = (
    dataset["Churn"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

target_distribution = pd.DataFrame({
    "count": target_counts,
    "percentage": target_percentages
})

target_distribution

,count,percentage
Churn,,
No,5174,73.46
Yes,1869,26.54


### Finding

The target variable is imbalanced.

Most customers belong to the `No` churn class, while the `Yes` churn class represents a smaller proportion of the dataset.

This means that accuracy alone will not be sufficient for evaluating the models. Metrics such as recall, precision, F1-score, balanced accuracy, and PR-AUC should also be considered, especially for the `Yes` class.

## Leakage Risks

This section identifies variables and preprocessing decisions that could introduce data leakage.

Data leakage occurs when the model receives information during training that would not be available at prediction time. This can produce unrealistically strong validation results and poor performance in production.

The analysis focuses on:

- target leakage;
- identifier columns;
- post-outcome variables;
- preprocessing before data splitting;
- duplicated or related observations across datasets.

In [179]:
potential_leakage_columns = [
    column
    for column in dataset.columns
    if any(
        keyword in column.lower()
        for keyword in [
            "churn",
            "cancel",
            "closed",
            "termination",
            "leave",
            "exit"
        ]
    )
]

potential_leakage_columns

['Churn']

In [180]:
customer_id_unique = (
    dataset["customerID"].nunique() == len(dataset)
)

print(f"customerID is unique: {customer_id_unique}")

customerID is unique: True


### Findings

- `Churn` is the target variable and must be excluded from the feature matrix.
- `customerID` is a unique identifier and should not be used as a predictive feature.
- No obvious post-churn or target-derived variables were found.
- Each customer appears only once, reducing the risk of the same entity appearing in both training and test sets.
- Preprocessing must be fitted only on training data. A scikit-learn `Pipeline` and `ColumnTransformer` will be used to prevent leakage during cross-validation and final model training.

### Decisions

1. Remove `Churn` from the feature matrix and use it only as the target.
2. Remove `customerID` before model training.
3. Perform the train/test split before fitting preprocessing transformations.
4. Place imputation, scaling, and encoding inside a scikit-learn pipeline.
5. Keep the test set untouched until final evaluation.

## Conclusion

The initial data audit identified no fully duplicated rows and no duplicated customer identifiers. Standard missing values were not detected; however, 11 blank values were found in the `TotalCharges` column.

All affected records have `tenure = 0`, which suggests that these customers have not yet accumulated historical charges. Based on this business interpretation, `TotalCharges` will be converted to a numeric data type and the corresponding missing values will be replaced with `0` during preprocessing.

The target variable, `Churn`, is imbalanced, with fewer positive churn cases than non-churn cases. Therefore, model evaluation should not rely on accuracy alone. Metrics such as recall, precision, F1-score, balanced accuracy, and PR-AUC should also be considered.

No obvious post-outcome variables were identified. However, `customerID` is a unique identifier and will be excluded from model training. All preprocessing operations, including imputation, encoding, and scaling, will be fitted only on the training data through a scikit-learn pipeline to reduce leakage risk.

Overall, the dataset is suitable for the next stage of the project after applying the documented preprocessing decisions.